# Lost in the Middle, revisited: free-tier reproduction

Reproduces the position experiment of *Lost in the Middle* (Liu et al., TACL 2024) on
Qwen2.5-3B-Instruct and Qwen3-4B-Instruct-2507, one model per T4. Plan and pre-registered
analysis: see the repository README.

**Kaggle settings:** Accelerator **GPU T4 x2** (not P100), Internet **on**.

**How to run:** set `STAGE` below, then *Save Version → Save & Run All (Commit)*.
Stages, in order: `smoke` → `pilot` → `qa` → `kv` → `rerun`. To resume or continue from an
earlier version, attach that version's output as an input (*Add Input → Your Work*); its
`results/` files are copied in and finished prompts are skipped.


In [ ]:
STAGE = "smoke"   # smoke | pilot | qa | kv | rerun | analyze
KV_LIMIT = 200    # key-value examples per position: 200 first; 500 if >5% of answers flip (plan §3)
MAX_MODEL_LEN = 20608  # longest measured prompt 20,458 tokens + 100 new tokens
ENFORCE_EAGER = False  # set True if CUDA-graph capture fails on the T4


In [ ]:
# Pre-registered guesses (plan §5): write these BEFORE the first qa/kv run and do not edit afterwards.
GUESSES = {
    "qwen2.5-3b":    {"qa20": "?", "kv300": "?"},   # U shape | Primacy only | Recency only | Flat
    "qwen3-4b-2507": {"qa20": "?", "kv300": "?"},
}


## Setup: environment, authors' data, resume

In [ ]:
import os; os.chdir('/kaggle/working')

## Pipeline code (generated from the repo's `litm/` package; edit there, not here)

In [ ]:
import os
for d in ['litm', 'litm/vendor', 'litm/vendor/lost_in_the_middle', 'litm/vendor/lost_in_the_middle/prompts']:
    os.makedirs(d, exist_ok=True)

In [ ]:
%%writefile litm/__init__.py


In [ ]:
%%writefile litm/analyze.py
"""Score saved outputs and produce the tables, verdicts and figures. Runs on CPU.

    python -m litm.analyze results/ --out analysis/
"""
import argparse
import json
from collections import defaultdict
from pathlib import Path

from litm.data import KV_MIDDLE, KV_POSITIONS, QA_MIDDLE, QA_POSITIONS
from litm.scoring import score
from litm.stats import accuracy_ci, verdict

# Paper reference curves, 20 documents, positions 1/5/10/15/20.
PAPER_QA20 = {
    "gpt-3.5-turbo-0613 (paper)": [75.8, 57.2, 53.8, 55.4, 63.2],
    "llama-2-70b-chat (authors' repo)": [56.8, 53.3, 54.1, 59.7, 69.5],
}
CURVES = {"qa20": (QA_POSITIONS, QA_MIDDLE), "kv300": (KV_POSITIONS, KV_MIDDLE)}


def load_scores(results_dir: Path, tag: str = "") -> dict:
    """{model: {(task, position): {idx: 0/1}}} from every <model>[__tag].jsonl file with this tag."""
    scores = defaultdict(lambda: defaultdict(dict))
    for path in sorted(results_dir.glob("*.jsonl")):
        if path.name.endswith(".prompt_samples.jsonl"):
            continue
        file_tag = path.stem.split("__")[1] if "__" in path.stem else ""
        if file_tag != tag:
            continue
        with open(path, encoding="utf-8") as f:
            for line in f:
                record = json.loads(line)
                scores[record["model"]][(record["task"], record["position"])][record["idx"]] = score(record)
    return scores


def summarize(scores: dict) -> dict:
    summary = {}
    for model, groups in scores.items():
        model_summary = {"accuracy": {}, "verdicts": {}}
        for (task, position), by_idx in sorted(groups.items(), key=lambda kv: (kv[0][0], kv[0][1] or 0)):
            name = task if position is None else f"{task}@{position}"
            model_summary["accuracy"][name] = accuracy_ci(list(by_idx.values()))
        for task, (positions, middle) in CURVES.items():
            if not all((task, p) in groups for p in positions):
                continue
            # Only questions present at every position, so contrasts stay paired.
            common = sorted(set.intersection(*(set(groups[(task, p)]) for p in positions)))
            first, mid, last = ([groups[(task, p)][i] for i in common] for p in (positions[0], middle, positions[-1]))
            result = verdict(first, mid, last)
            result["n"] = len(common)
            model_summary["verdicts"][task] = result
        summary[model] = model_summary
    return summary


def compare_rerun(main: dict, rerun: dict, tolerance: float = 0.02, min_agreement: float = 0.97) -> str:
    """Plan section 8: per position, rerun accuracy within +/-2 points and >=97% identical scores."""
    lines = ["| Model | Setting | Main | Rerun | Agreement | Pass |", "|---|---|---|---|---|---|"]
    all_pass = True
    for model, groups in rerun.items():
        for (task, position), by_idx in sorted(groups.items(), key=lambda kv: (kv[0][0], kv[0][1] or 0)):
            reference = main.get(model, {}).get((task, position), {})
            common = sorted(set(by_idx) & set(reference))
            if not common:
                continue
            a = sum(reference[i] for i in common) / len(common)
            b = sum(by_idx[i] for i in common) / len(common)
            agreement = sum(reference[i] == by_idx[i] for i in common) / len(common)
            ok = abs(a - b) <= tolerance and agreement >= min_agreement
            all_pass &= ok
            lines.append(f"| {model} | {task}@{position} (n={len(common)}) | {100 * a:.1f} | {100 * b:.1f} | "
                         f"{100 * agreement:.1f}% | {'yes' if ok else 'NO'} |")
    lines.append(f"\nRerun check: {'PASS' if all_pass else 'FAIL'}")
    return "\n".join(lines)


def to_markdown(summary: dict) -> str:
    lines = []
    for model, s in summary.items():
        lines += [f"## {model}", "", "| Setting | Accuracy | 95% CI | n |", "|---|---|---|---|"]
        for name, a in s["accuracy"].items():
            lines.append(f"| {name} | {100 * a['acc']:.1f} | {100 * a['low']:.1f}–{100 * a['high']:.1f} | {a['n']} |")
        for task, v in s["verdicts"].items():
            lines += ["", f"**{task} verdict: {v['shape']}** (n = {v['n']})", ""]
            for label in ("first_vs_middle", "last_vs_middle"):
                c = v[label]
                lines.append(
                    f"- {label.replace('_', ' ')}: {100 * c['diff']:+.1f} pts "
                    f"(95% CI {100 * c['low']:+.1f} to {100 * c['high']:+.1f}), McNemar p = {c['p']:.3g}, "
                    f"discordance {100 * c['discordance']:.1f}%, MDE {100 * c['mde']:.1f} pts"
                )
        lines.append("")
    return "\n".join(lines)


def plot(summary: dict, out_dir: Path) -> None:
    import matplotlib

    matplotlib.use("Agg")
    import matplotlib.pyplot as plt

    for task, (positions, _) in CURVES.items():
        fig, ax = plt.subplots(figsize=(6, 4))
        x = [p + 1 for p in positions]
        for model, s in summary.items():
            points = [s["accuracy"].get(f"{task}@{p}") for p in positions]
            if not all(points):
                continue
            y = [100 * a["acc"] for a in points]
            err = [[100 * (a["acc"] - a["low"]) for a in points], [100 * (a["high"] - a["acc"]) for a in points]]
            line = ax.errorbar(x, y, yerr=err, marker="o", capsize=3, label=model)
            if task == "qa20":
                for baseline, style in (("closedbook", ":"), ("oracle", "--")):
                    if baseline in s["accuracy"]:
                        ax.axhline(100 * s["accuracy"][baseline]["acc"], ls=style, color=line[0].get_color(), alpha=0.6,
                                   label=f"{model} {baseline}")
        if task == "qa20":
            for name, y in PAPER_QA20.items():
                ax.plot(x, y, marker=".", color="grey", alpha=0.5, ls="-.", label=name)
        ax.set_xticks(x)
        ax.set_xlabel("Position of the answer" if task == "qa20" else "Position of the key")
        ax.set_ylabel("Accuracy (%)")
        ax.set_title("20 documents" if task == "qa20" else "300 key-value pairs")
        ax.legend(fontsize=7)
        fig.tight_layout()
        fig.savefig(out_dir / f"{task}.png", dpi=150)
        plt.close(fig)


def main(argv=None):
    parser = argparse.ArgumentParser()
    parser.add_argument("results", type=Path)
    parser.add_argument("--out", type=Path, default=Path("analysis"))
    parser.add_argument("--tag", default="", help="Analyse <model>__<tag>.jsonl files instead of the main runs")
    parser.add_argument("--no-plot", action="store_true")
    args = parser.parse_args(argv)

    args.out.mkdir(parents=True, exist_ok=True)
    if args.tag == "rerun":
        report = compare_rerun(load_scores(args.results), load_scores(args.results, "rerun"))
        (args.out / "rerun_check.md").write_text(report, encoding="utf-8")
        print(report)
        return
    summary = summarize(load_scores(args.results, args.tag))
    (args.out / "summary.json").write_text(json.dumps(summary, indent=2, default=float))
    report = to_markdown(summary)
    (args.out / "summary.md").write_text(report, encoding="utf-8")
    print(report)
    if not args.no_plot:
        plot(summary, args.out)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile litm/data.py
"""Load the authors' data files and turn them into prompt items.

Every item is identified by (task, position, idx), where idx is the line number in the
authors' file. The files have no ID field, so `check_qa_alignment` confirms that line i
is the same question in every file.
"""
import json
from copy import deepcopy
from pathlib import Path
from typing import Dict, Iterator, List

from xopen import xopen

from litm.vendor.lost_in_the_middle.prompting import (
    Document,
    get_closedbook_qa_prompt,
    get_kv_retrieval_prompt,
    get_qa_prompt,
)

# Pinned commit of github.com/nelson-liu/lost-in-the-middle (data re-uploaded 2024-01-04).
AUTHORS_COMMIT = "29b8a6d042ce29abccee3db1a73171a107d7e6af"

QA_POSITIONS = [0, 4, 9, 14, 19]  # 0-based; paper positions 1, 5, 10, 15, 20
KV_POSITIONS = [0, 74, 149, 224, 299]  # 0-based; positions 1, 75, 150, 225, 300
QA_MIDDLE = 9
KV_MIDDLE = 149


def read_jsonl(path) -> List[dict]:
    with xopen(str(path)) as f:
        return [json.loads(line) for line in f]


def qa_path(root: Path, gold_index: int) -> Path:
    return root / "qa_data" / "20_total_documents" / f"nq-open-20_total_documents_gold_at_{gold_index}.jsonl.gz"


def oracle_path(root: Path) -> Path:
    return root / "qa_data" / "nq-open-oracle.jsonl.gz"


def kv_path(root: Path, n_keys: int = 300) -> Path:
    return root / "kv_retrieval_data" / f"kv-retrieval-{n_keys}_keys.jsonl.gz"


def check_qa_alignment(root: Path) -> int:
    """Raise if any 20-doc file disagrees with the oracle file on question, answers or gold slot."""
    oracle = read_jsonl(oracle_path(root))
    for gold_index in QA_POSITIONS:
        examples = read_jsonl(qa_path(root, gold_index))
        if len(examples) != len(oracle):
            raise ValueError(f"gold_at_{gold_index}: {len(examples)} examples, oracle has {len(oracle)}")
        for idx, (example, reference) in enumerate(zip(examples, oracle)):
            if example["question"] != reference["question"] or example["answers"] != reference["answers"]:
                raise ValueError(f"gold_at_{gold_index} line {idx} does not match the oracle file")
            gold_slots = [i for i, ctx in enumerate(example["ctxs"]) if ctx.get("isgold")]
            if gold_slots != [gold_index]:
                raise ValueError(f"gold_at_{gold_index} line {idx}: gold document at {gold_slots}")
    return len(oracle)


def qa_items(root: Path, task: str, position=None, query_aware: bool = False) -> Iterator[Dict]:
    """task is 'closedbook', 'oracle' or 'qa20'; position is the 0-based gold index for 'qa20'."""
    path = qa_path(root, position) if task == "qa20" else oracle_path(root)
    for idx, example in enumerate(read_jsonl(path)):
        if task == "closedbook":
            prompt = get_closedbook_qa_prompt(example["question"])
        else:
            documents = [Document.from_dict(ctx) for ctx in example["ctxs"]]
            prompt = get_qa_prompt(
                example["question"],
                documents,
                mention_random_ordering=False,
                query_aware_contextualization=query_aware,
            )
        yield {
            "task": task,
            "position": position,
            "idx": idx,
            "prompt": prompt,
            "answers": example["answers"],
        }


def kv_items(root: Path, position: int, n_keys: int = 300, limit=None, query_aware: bool = False) -> Iterator[Dict]:
    """Move the gold pair to `position`, as the authors' get_kv_responses_* scripts do."""
    for idx, example in enumerate(read_jsonl(kv_path(root, n_keys))):
        if limit is not None and idx >= limit:
            break
        records = deepcopy(example["ordered_kv_records"])
        gold = records.pop(records.index([example["key"], example["value"]]))
        records.insert(position, gold)
        yield {
            "task": f"kv{n_keys}",
            "position": position,
            "idx": idx,
            "prompt": get_kv_retrieval_prompt(records, example["key"], query_aware_contextualization=query_aware),
            "answers": [example["value"]],
        }


def all_items(root: Path, tasks: List[str], kv_limit=None) -> List[Dict]:
    """Build every item for the named tasks: closedbook, oracle, qa20, kv300."""
    items = []
    for task in tasks:
        if task in ("closedbook", "oracle"):
            items.extend(qa_items(root, task))
        elif task == "qa20":
            for position in QA_POSITIONS:
                items.extend(qa_items(root, task, position))
        elif task == "kv300":
            for position in KV_POSITIONS:
                items.extend(kv_items(root, position, 300, limit=kv_limit))
        else:
            raise ValueError(f"Unknown task: {task}")
    return items


In [ ]:
%%writefile litm/run.py
"""Run one model on a set of tasks with vLLM, saving every output as it goes.

Usage (one model per GPU):
    CUDA_VISIBLE_DEVICES=0 python -m litm.run --model qwen2.5-3b --tasks closedbook oracle qa20 --data <authors repo> --out results
    CUDA_VISIBLE_DEVICES=1 python -m litm.run --model qwen3-4b-2507 --tasks kv300 --kv-limit 200 ...

Rerunning the same command resumes: finished (task, position, idx) triples are skipped.
"""
import argparse
import hashlib
import json
import platform
import sys
import time
from pathlib import Path

from litm.data import all_items

MODELS = {
    "qwen2.5-3b": {"hf_id": "Qwen/Qwen2.5-3B-Instruct", "revision": "aa8e72537993ba99e69dfaafa59ed015b17504d1"},
    "qwen3-4b-2507": {"hf_id": "Qwen/Qwen3-4B-Instruct-2507", "revision": "cdbee75f17c01a7cc42f958dc650907174af0554"},
}
MAX_NEW_TOKENS = 100


def key(record) -> tuple:
    return (record["task"], record["position"], record["idx"])


def load_done(path: Path) -> set:
    if not path.exists():
        return set()
    with open(path, encoding="utf-8") as f:
        return {key(json.loads(line)) for line in f if line.strip()}


def render(tokenizer, prompt: str) -> str:
    """The paper prompt as one user message in the model's own chat template, as shipped."""
    return tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt}], tokenize=False, add_generation_prompt=True
    )


def environment() -> dict:
    import torch
    import transformers
    import vllm

    return {
        "python": sys.version.split()[0],
        "platform": platform.platform(),
        "vllm": vllm.__version__,
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "gpu": torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
    }


def main(argv=None):
    parser = argparse.ArgumentParser()
    parser.add_argument("--model", choices=sorted(MODELS), required=True)
    parser.add_argument("--tasks", nargs="+", required=True, help="closedbook oracle qa20 kv300")
    parser.add_argument("--data", type=Path, required=True, help="Path to the authors' repo checkout")
    parser.add_argument("--out", type=Path, required=True)
    parser.add_argument("--kv-limit", type=int, default=None, help="Use only the first N key-value examples")
    parser.add_argument("--qa-limit", type=int, default=None, help="Use only the first N questions (pilot/rerun)")
    parser.add_argument("--batch-size", type=int, default=512)
    parser.add_argument("--max-model-len", type=int, default=20608)  # longest prompt measured: 20,458 tokens (kv300) + 100 new tokens
    parser.add_argument("--enforce-eager", action="store_true")
    parser.add_argument("--tag", default="", help="Suffix for the output file, e.g. 'pilot' or 'rerun'")
    args = parser.parse_args(argv)

    from transformers import AutoTokenizer
    from vllm import LLM, SamplingParams
    from vllm.inputs import TokensPrompt

    model = MODELS[args.model]
    args.out.mkdir(parents=True, exist_ok=True)
    stem = f"{args.model}{'__' + args.tag if args.tag else ''}"
    out_path = args.out / f"{stem}.jsonl"
    meta_path = args.out / f"{stem}.meta.json"
    samples_path = args.out / f"{stem}.prompt_samples.jsonl"

    items = all_items(args.data, args.tasks, kv_limit=args.kv_limit)
    if args.qa_limit is not None:
        items = [x for x in items if x["task"].startswith("kv") or x["idx"] < args.qa_limit]
    done = load_done(out_path)
    pending = [x for x in items if key(x) not in done]
    print(f"{len(items)} items, {len(done)} already done, {len(pending)} to run", flush=True)
    if not pending:
        return

    # Render and tokenize everything first so over-long prompts fail before any GPU time is spent.
    tokenizer = AutoTokenizer.from_pretrained(model["hf_id"], revision=model["revision"])
    too_long = []
    seen_groups = set()
    rendered_all = [render(tokenizer, item["prompt"]) for item in pending]
    token_ids_all = tokenizer(rendered_all, add_special_tokens=False)["input_ids"]  # batched = parallel
    with open(samples_path, "a", encoding="utf-8") as samples:
        for item, rendered, token_ids in zip(pending, rendered_all, token_ids_all):
            item["token_ids"] = token_ids
            item["prompt_sha256"] = hashlib.sha256(rendered.encode("utf-8")).hexdigest()
            if len(item["token_ids"]) + MAX_NEW_TOKENS > args.max_model_len:
                too_long.append((key(item), len(item["token_ids"])))
            group = (item["task"], item["position"])
            if group not in seen_groups:  # one full rendered prompt per (task, position)
                seen_groups.add(group)
                samples.write(json.dumps({**{k: item[k] for k in ("task", "position", "idx")}, "rendered": rendered}) + "\n")
    if too_long:
        raise SystemExit(f"{len(too_long)} prompts exceed --max-model-len {args.max_model_len}, e.g. {too_long[:3]}")
    lengths = [len(x["token_ids"]) for x in pending]
    print(f"prompt tokens: max {max(lengths)}, mean {sum(lengths) / len(lengths):.0f}, total {sum(lengths)}", flush=True)

    llm = LLM(
        model=model["hf_id"],
        revision=model["revision"],
        dtype="float16",  # T4 has no bfloat16
        attention_backend="TRITON_ATTN",  # FlashAttention needs sm80+; FlashInfer has crashed on T4
        max_model_len=args.max_model_len,
        gpu_memory_utilization=0.9,
        enforce_eager=args.enforce_eager,
        seed=0,
    )
    sampling = SamplingParams(temperature=0.0, max_tokens=MAX_NEW_TOKENS)

    meta = {
        "model": args.model,
        **model,
        "tasks": args.tasks,
        "kv_limit": args.kv_limit,
        "qa_limit": args.qa_limit,
        "max_model_len": args.max_model_len,
        "enforce_eager": args.enforce_eager,
        "max_new_tokens": MAX_NEW_TOKENS,
        "environment": environment(),
    }
    meta_path.write_text(json.dumps(meta, indent=2))

    started, tokens_done = time.time(), 0
    with open(out_path, "a", encoding="utf-8") as out:
        for start in range(0, len(pending), args.batch_size):
            batch = pending[start : start + args.batch_size]
            results = llm.generate([TokensPrompt(prompt_token_ids=x["token_ids"]) for x in batch], sampling, use_tqdm=False)
            for item, result in zip(batch, results):
                completion = result.outputs[0]
                record = {k: item[k] for k in ("task", "position", "idx", "answers", "prompt_sha256")}
                record.update(
                    model=args.model,
                    n_prompt_tokens=len(item["token_ids"]),
                    output=completion.text,
                    n_output_tokens=len(completion.token_ids),
                    finish_reason=completion.finish_reason,
                )
                out.write(json.dumps(record, ensure_ascii=False) + "\n")
            out.flush()
            tokens_done += sum(len(x["token_ids"]) for x in batch)
            elapsed = time.time() - started
            print(
                f"{start + len(batch)}/{len(pending)} done, {elapsed / 60:.1f} min, {tokens_done / elapsed:.0f} prompt tok/s",
                flush=True,
            )


if __name__ == "__main__":
    main()


In [ ]:
%%writefile litm/scoring.py
"""Scoring rules copied from the authors' evaluate_*_responses.py scripts.

One deliberate difference: we strip leading whitespace before taking the first line.
The authors' vLLM script strips generations before scoring, but their evaluate script
splits first, so an answer starting with a newline would score 0.
"""
from typing import List

from litm.vendor.lost_in_the_middle.metrics import best_subspan_em


def score_qa(output: str, answers: List[str]) -> int:
    first_line = output.strip().split("\n")[0].strip()
    return int(best_subspan_em(prediction=first_line, ground_truths=answers))


def score_kv(output: str, value: str) -> int:
    return int(value.lower() in output.lower())


def score(record: dict) -> int:
    if record["task"].startswith("kv"):
        return score_kv(record["output"], record["answers"][0])
    return score_qa(record["output"], record["answers"])


In [ ]:
%%writefile litm/stats.py
"""Paired statistics for position contrasts (plan section 5).

All comparisons are paired: a[i] and b[i] are 0/1 scores for the same question at two
positions.
"""
from math import comb, sqrt
from statistics import NormalDist
from typing import Dict, List, Sequence

import numpy as np

N_BOOTSTRAP = 10_000
SEED = 0
# Pre-registered equivalence margin: a contrast counts as "no effect" only if its whole
# 95% CI lies within +/- 3 points. (Using the MDE here would call tiny samples "Flat",
# because the MDE widens as fast as the CI.)
FLAT_MARGIN = 0.03


def accuracy_ci(scores: Sequence[int], n_boot: int = N_BOOTSTRAP, seed: int = SEED) -> Dict:
    x = np.asarray(scores, dtype=float)
    rng = np.random.default_rng(seed)
    means = x[rng.integers(0, len(x), size=(n_boot, len(x)))].mean(axis=1)
    low, high = np.percentile(means, [2.5, 97.5])
    return {"acc": x.mean(), "low": low, "high": high, "n": len(x)}


def paired_bootstrap(a: Sequence[int], b: Sequence[int], n_boot: int = N_BOOTSTRAP, seed: int = SEED) -> Dict:
    """95% CI for mean(a) - mean(b), resampling questions."""
    diff = np.asarray(a, dtype=float) - np.asarray(b, dtype=float)
    rng = np.random.default_rng(seed)
    means = diff[rng.integers(0, len(diff), size=(n_boot, len(diff)))].mean(axis=1)
    low, high = np.percentile(means, [2.5, 97.5])
    return {"diff": diff.mean(), "low": low, "high": high}


def mcnemar_exact(a: Sequence[int], b: Sequence[int]) -> Dict:
    """Two-sided exact McNemar test on discordant pairs."""
    a, b = np.asarray(a), np.asarray(b)
    only_a = int(((a == 1) & (b == 0)).sum())
    only_b = int(((a == 0) & (b == 1)).sum())
    n = only_a + only_b
    if n == 0:
        p = 1.0
    else:
        tail = sum(comb(n, k) for k in range(min(only_a, only_b) + 1)) / 2**n
        p = min(1.0, 2 * tail)
    return {"only_a": only_a, "only_b": only_b, "discordance": n / len(a), "p": p}


def holm(pvalues: List[float], alpha: float = 0.05) -> List[bool]:
    """Holm step-down: which hypotheses are rejected."""
    order = sorted(range(len(pvalues)), key=lambda i: pvalues[i])
    rejected = [False] * len(pvalues)
    for rank, i in enumerate(order):
        if pvalues[i] > alpha / (len(pvalues) - rank):
            break
        rejected[i] = True
    return rejected


def mde(discordance: float, n: int, alpha: float = 0.025, power: float = 0.8) -> float:
    """Minimum detectable paired difference (Card et al. 2020 approximation).

    alpha defaults to 0.05 / 2, the stricter Holm threshold for two contrasts.
    """
    z = NormalDist().inv_cdf(1 - alpha / 2) + NormalDist().inv_cdf(power)
    return z * sqrt(discordance / n)


def contrast(a: Sequence[int], b: Sequence[int]) -> Dict:
    result = {**paired_bootstrap(a, b), **mcnemar_exact(a, b)}
    result["mde"] = mde(result["discordance"], len(a))
    return result


def verdict(first: Sequence[int], middle: Sequence[int], last: Sequence[int]) -> Dict:
    """Pre-registered shape verdict from the first-vs-middle and last-vs-middle contrasts."""
    first_vs_middle = contrast(first, middle)
    last_vs_middle = contrast(last, middle)
    first_sig, last_sig = holm([first_vs_middle["p"], last_vs_middle["p"]])
    # A significant contrast only counts toward the shape if the middle is the lower one.
    first_up = first_sig and first_vs_middle["diff"] > 0
    last_up = last_sig and last_vs_middle["diff"] > 0

    if first_up and last_up:
        shape = "U shape"
    elif first_up and not last_sig:
        shape = "Primacy only"
    elif last_up and not first_sig:
        shape = "Recency only"
    elif not first_sig and not last_sig:
        within = all(
            -FLAT_MARGIN <= c["low"] and c["high"] <= FLAT_MARGIN for c in (first_vs_middle, last_vs_middle)
        )
        shape = "Flat" if within else "Inconclusive"
    else:
        shape = "Other"  # e.g. the middle is significantly *better* than an end
    return {"shape": shape, "first_vs_middle": first_vs_middle, "last_vs_middle": last_vs_middle}


In [ ]:
%%writefile litm/vendor/__init__.py


In [ ]:
%%writefile litm/vendor/lost_in_the_middle/__init__.py


In [ ]:
%%writefile litm/vendor/lost_in_the_middle/LICENSE
MIT License

Copyright (c) 2023 Nelson Liu

Permission is hereby granted, free of charge, to any person obtaining a copy
of this software and associated documentation files (the "Software"), to deal
in the Software without restriction, including without limitation the rights
to use, copy, modify, merge, publish, distribute, sublicense, and/or sell
copies of the Software, and to permit persons to whom the Software is
furnished to do so, subject to the following conditions:

The above copyright notice and this permission notice shall be included in all
copies or substantial portions of the Software.

THE SOFTWARE IS PROVIDED "AS IS", WITHOUT WARRANTY OF ANY KIND, EXPRESS OR
IMPLIED, INCLUDING BUT NOT LIMITED TO THE WARRANTIES OF MERCHANTABILITY,
FITNESS FOR A PARTICULAR PURPOSE AND NONINFRINGEMENT. IN NO EVENT SHALL THE
AUTHORS OR COPYRIGHT HOLDERS BE LIABLE FOR ANY CLAIM, DAMAGES OR OTHER
LIABILITY, WHETHER IN AN ACTION OF CONTRACT, TORT OR OTHERWISE, ARISING FROM,
OUT OF OR IN CONNECTION WITH THE SOFTWARE OR THE USE OR OTHER DEALINGS IN THE
SOFTWARE.


In [ ]:
%%writefile litm/vendor/lost_in_the_middle/metrics.py
#!/usr/bin/env python3
import string
from typing import List

import regex


def normalize_answer(s: str) -> str:
    """Normalization from the SQuAD evaluation script.

    See https://worksheets.codalab.org/rest/bundles/0x6b567e1cf2e041ec80d7098f031c5c9e/contents/blob/
    """

    def remove_articles(text):
        return regex.sub(r"\b(a|an|the)\b", " ", text)

    def white_space_fix(text):
        return " ".join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return "".join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(s))))


def best_subspan_em(prediction: str, ground_truths: List[str]) -> float:
    normalized_prediction = normalize_answer(prediction)

    for ground_truth in ground_truths:
        normalized_ground_truth = normalize_answer(ground_truth)
        if normalized_ground_truth.lower() in normalized_prediction.lower():
            return 1.0
    return 0.0


In [ ]:
%%writefile litm/vendor/lost_in_the_middle/prompting.py
#!/usr/bin/env python3
import pathlib
from copy import deepcopy
from typing import List, Optional, Tuple, Type, TypeVar

from pydantic.dataclasses import dataclass

PROMPTS_ROOT = (pathlib.Path(__file__).parent / "prompts").resolve()

T = TypeVar("T")


@dataclass(frozen=True)
class Document:
    title: str
    text: str
    id: Optional[str] = None
    score: Optional[float] = None
    hasanswer: Optional[bool] = None
    isgold: Optional[bool] = None
    original_retrieval_index: Optional[int] = None

    @classmethod
    def from_dict(cls: Type[T], data: dict) -> T:
        data = deepcopy(data)
        if not data:
            raise ValueError("Must provide data for creation of Document from dict.")
        id = data.pop("id", None)
        score = data.pop("score", None)
        # Convert score to float if it's provided.
        if score is not None:
            score = float(score)
        return cls(**dict(data, id=id, score=score))


def get_qa_prompt(
    question: str, documents: List[Document], mention_random_ordering: bool, query_aware_contextualization: bool
):
    if not question:
        raise ValueError(f"Provided `question` must be truthy, got: {question}")
    if not documents:
        raise ValueError(f"Provided `documents` must be truthy, got: {documents}")

    if mention_random_ordering and query_aware_contextualization:
        raise ValueError("Mentioning random ordering cannot be currently used with query aware contextualization")

    if mention_random_ordering:
        prompt_filename = "qa_ordered_randomly.prompt"
    elif query_aware_contextualization:
        prompt_filename = "qa_with_query_aware_contextualization.prompt"
    else:
        prompt_filename = "qa.prompt"

    with open(PROMPTS_ROOT / prompt_filename) as f:
        prompt_template = f.read().rstrip("\n")

    # Format the documents into strings
    formatted_documents = []
    for document_index, document in enumerate(documents):
        formatted_documents.append(f"Document [{document_index+1}](Title: {document.title}) {document.text}")
    return prompt_template.format(question=question, search_results="\n".join(formatted_documents))


def get_closedbook_qa_prompt(question: str):
    if not question:
        raise ValueError(f"Provided `question` must be truthy, got: {question}")
    with open(PROMPTS_ROOT / "closedbook_qa.prompt") as f:
        prompt_template = f.read().rstrip("\n")

    return prompt_template.format(question=question)


def get_kv_retrieval_prompt(
    data: List[Tuple[str, str]],
    key: str,
    query_aware_contextualization: bool = False,
):
    if not data:
        raise ValueError(f"Provided `data` must be truthy, got: {data}")
    if not key:
        raise ValueError(f"Provided `key` must be truthy, got: {key}")
    if key not in [x[0] for x in data]:
        raise ValueError(f"Did not find provided `key` {key} in data {data}")
    if len(data) != len(set([x[0] for x in data])):
        raise ValueError(f"`data` has duplicate keys: {data}")
    if len(data) < 2:
        raise ValueError(f"Must have at least 2 items in data: {data}")

    if query_aware_contextualization:
        with open(PROMPTS_ROOT / "kv_retrieval_with_query_aware_contextualization.prompt") as f:
            prompt_template = f.read().rstrip("\n")
    else:
        with open(PROMPTS_ROOT / "kv_retrieval.prompt") as f:
            prompt_template = f.read().rstrip("\n")

    # Format the KV data into a string
    formatted_kv_records = ""
    for index, record in enumerate(data):
        start_character = "{" if index == 0 else " "
        data_string = f'"{record[0]}": "{record[1]}"'
        end_character = ",\n" if index != len(data) - 1 else "}"
        formatted_kv_records += start_character + data_string + end_character

    return prompt_template.format(formatted_kv_records=formatted_kv_records, key=key)


In [ ]:
%%writefile litm/vendor/lost_in_the_middle/prompts/closedbook_qa.prompt
Question: {question}
Answer:


In [ ]:
%%writefile litm/vendor/lost_in_the_middle/prompts/kv_retrieval.prompt
Extract the value corresponding to the specified key in the JSON object below.

JSON data:
{formatted_kv_records}

Key: "{key}"
Corresponding value:


In [ ]:
%%writefile litm/vendor/lost_in_the_middle/prompts/kv_retrieval_with_query_aware_contextualization.prompt
Extract the value corresponding to the specified key in the JSON object below.

Key: "{key}"

JSON data:
{formatted_kv_records}

Key: "{key}"
Corresponding value:


In [ ]:
%%writefile litm/vendor/lost_in_the_middle/prompts/qa.prompt
Write a high-quality answer for the given question using only the provided search results (some of which might be irrelevant).

{search_results}

Question: {question}
Answer:

In [ ]:
%%writefile litm/vendor/lost_in_the_middle/prompts/qa_ordered_randomly.prompt
Write a high-quality answer for the given question using only the provided search results (some of which might be irrelevant). The search results are ordered randomly.

{search_results}

Question: {question}
Answer:

In [ ]:
%%writefile litm/vendor/lost_in_the_middle/prompts/qa_with_query_aware_contextualization.prompt
Write a high-quality answer for the given question using only the provided search results (some of which might be irrelevant).

Question: {question}

{search_results}

Question: {question}
Answer:


In [ ]:
import glob, os, shutil, subprocess, sys, time
WORK = "/kaggle/working"
VENV = "/tmp/vllm-env"
AUTHORS = "/tmp/lost-in-the-middle"
AUTHORS_COMMIT = "29b8a6d042ce29abccee3db1a73171a107d7e6af"
os.chdir(WORK)

def sh(cmd):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

# Isolated environment for vLLM 0.18.1 (it pins its own torch 2.10.0); this kernel never imports torch.
if not os.path.exists(f"{VENV}/bin/python"):
    sh("pip install -q uv")
    sh(f"uv venv -q {VENV} --python 3.12")
    sh(f"uv pip install -q --python {VENV}/bin/python vllm==0.18.1 pydantic regex xopen numpy matplotlib")
PY = f"{VENV}/bin/python"

# The authors' data and code at a pinned commit.
if not os.path.exists(AUTHORS):
    sh(f"git init -q {AUTHORS} && cd {AUTHORS} && git fetch -q --depth 1 "
       f"https://github.com/nelson-liu/lost-in-the-middle.git {AUTHORS_COMMIT} && git checkout -q FETCH_HEAD")

# Resume: copy results from any attached earlier version of this notebook.
os.makedirs("results", exist_ok=True)
for path in glob.glob("/kaggle/input/**/results/*.jsonl", recursive=True):
    target = os.path.join("results", os.path.basename(path))
    if not os.path.exists(target):
        shutil.copy(path, target)
        print("resumed", target)
sh("nvidia-smi --query-gpu=name,memory.total --format=csv")


## Run the stage (one model per GPU)

In [ ]:
STAGES = {
    # stage: (extra args, output tag, tasks)
    "smoke": (["--qa-limit", "2", "--kv-limit", "1"], "smoke", ["closedbook", "oracle", "qa20", "kv300"]),
    "pilot": (["--qa-limit", "20", "--kv-limit", "4"], "pilot", ["closedbook", "oracle", "qa20", "kv300"]),
    "qa":    ([], "", ["closedbook", "oracle", "qa20"]),
    "kv":    (["--kv-limit", str(KV_LIMIT)], "", ["kv300"]),
    "rerun": (["--qa-limit", "300", "--kv-limit", "50"], "rerun", ["qa20", "kv300"]),
}
MODELS = ["qwen2.5-3b", "qwen3-4b-2507"]  # model i runs on GPU i

if STAGE in STAGES:
    extra, tag, tasks = STAGES[STAGE]
    procs = []
    for gpu, model in enumerate(MODELS):
        cmd = [PY, "-m", "litm.run", "--model", model, "--tasks", *tasks, "--data", AUTHORS,
               "--out", "results", "--max-model-len", str(MAX_MODEL_LEN), *extra]
        if tag:
            cmd += ["--tag", tag]
        if ENFORCE_EAGER:
            cmd.append("--enforce-eager")
        log = open(f"results/{model}{'__' + tag if tag else ''}.{STAGE}.log", "a")
        env = {**os.environ, "CUDA_VISIBLE_DEVICES": str(gpu), "PYTHONPATH": WORK}
        procs.append((model, log.name, subprocess.Popen(cmd, stdout=log, stderr=subprocess.STDOUT, env=env)))
        print(f"GPU {gpu}: {model} -> {log.name}")
    started = time.time()
    while any(p.poll() is None for _, _, p in procs):
        time.sleep(300)
        print(f"--- {(time.time() - started) / 60:.0f} min")
        for model, log_name, _ in procs:
            lines = open(log_name).read().splitlines()
            print(model, "|", lines[-1] if lines else "")
    for model, log_name, p in procs:
        print(f"{model}: exit code {p.returncode}")
        if p.returncode != 0:
            print("".join(open(log_name).readlines()[-40:]))


In [ ]:
# Smoke/pilot check: read the outputs by hand before any full run.
import json
if STAGE in ("smoke", "pilot"):
    for model in MODELS:
        path = f"results/{model}__{STAGE}.jsonl"
        if not os.path.exists(path):
            continue
        print("=" * 30, model)
        for line in open(path):
            r = json.loads(line)
            if STAGE == "smoke" or r["idx"] < 2:
                print(f"{r['task']}@{r['position']} #{r['idx']} ({r['n_prompt_tokens']} tok, {r['finish_reason']}): "
                      f"{r['output'][:120]!r}  gold={r['answers'][:2]}")


## Score and analyse

In [ ]:
tag_args = ["--tag", STAGES[STAGE][1]] if STAGE in STAGES and STAGES[STAGE][1] else []
sh(" ".join([f"PYTHONPATH={WORK}", PY, "-m", "litm.analyze", "results", "--out", "analysis", *tag_args]))
